Copyright 2026 Snowflake Inc.
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

# Exercise 2.2: Branching, Tagging, and Time Travel

In this exercise, you'll learn how to use Apache Iceberg's versioning features with the NYC Yellow Taxi dataset:
- **Snapshots**: Every write creates a new version
- **Time Travel**: Query historical versions of your table
- **Tagging**: Mark important snapshots with memorable names
- **Rollback**: Recover from mistakes by reverting to previous states
- **Write-Audit-Publish (WAP)**: Stage changes for review before making them visible
- **Branching and Merging**: Create isolated branches for experimentation

## Learning Objectives
- Understand how snapshots track table history
- Use time travel to query past versions
- Create and use tags for important milestones
- Rollback to previous table states
- Use Write-Audit-Publish to stage and review changes before publishing
- Create branches, write to them in isolation, and merge back to main

⚠️ **IMPORTANT**: This environment uses **Spark Connect** — a single Spark server runs in the background and all notebooks connect to it as lightweight clients (no per-notebook JVM). If you see `ConnectionRefusedError: [Errno 111] Connection refused` when initializing the Spark Session, the Spark Connect server may not be running. Check the Docker container logs with `docker logs jupyter-spark`. If the server failed to start, try restarting the container with `docker compose restart jupyter`.

## Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = SparkSession.builder \
    .appName("BranchingAndTagging") \
    .getOrCreate()

print(f"Spark {spark.version} initialized!")

## Create Namespace

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.timetravel")
print("Namespace 'timetravel' created!")

## Download NYC Taxi Data

We'll use NYC Yellow Taxi trip data for **June through August 2023**. If you've already run Exercise 2.1, the files are already in MinIO and this step will skip the download.

In [ ]:
import boto3
from botocore.client import Config
import urllib.request
import os

s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='password',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-{:02d}.parquet"
bucket = "warehouse"

for month in [6, 7, 8]:
    filename = f"yellow_tripdata_2023-{month:02d}.parquet"
    key = f"raw/{filename}"
    try:
        s3_client.head_object(Bucket=bucket, Key=key)
        print(f"{filename} already in MinIO, skipping download")
    except:
        local_path = f"/tmp/{filename}"
        print(f"Downloading {filename} (~45MB)...")
        urllib.request.urlretrieve(base_url.format(month), local_path)
        s3_client.upload_file(local_path, bucket, key)
        os.remove(local_path)
        print(f"  Uploaded to s3a://{bucket}/{key}")

print("\nAll taxi data ready in MinIO!")

## Part 1: Create a Table and Build History

We'll build up table history by loading data in stages, then cleaning it, creating three distinct snapshots.

### Create Table from June 2023

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.timetravel.nyc_taxi")

spark.sql("""
    CREATE TABLE polaris.timetravel.nyc_taxi
    USING iceberg
    AS SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
""")

count_1 = spark.sql("SELECT COUNT(*) FROM polaris.timetravel.nyc_taxi").collect()[0][0]
print(f"Snapshot 1: Loaded June 2023, {count_1:,} trips")

### Insert July 2023

In [ ]:
spark.sql("""
    INSERT INTO polaris.timetravel.nyc_taxi
    SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-07.parquet`
""")

count_2 = spark.sql("SELECT COUNT(*) FROM polaris.timetravel.nyc_taxi").collect()[0][0]
print(f"Snapshot 2: Added July 2023, {count_2:,} total trips")

### Remove Negative Fares

In [ ]:
bad_fares = spark.sql("""
    SELECT COUNT(*) FROM polaris.timetravel.nyc_taxi WHERE fare_amount <= 0
""").collect()[0][0]

spark.sql("""
    DELETE FROM polaris.timetravel.nyc_taxi WHERE fare_amount <= 0
""")

count_3 = spark.sql("SELECT COUNT(*) FROM polaris.timetravel.nyc_taxi").collect()[0][0]
print(f"Snapshot 3: Cleaned bad fares. {count_3:,} trips remaining ({bad_fares:,} removed)")

## Part 2: View Snapshot History

In [ ]:
print("Snapshot history:")
snapshots_df = spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM polaris.timetravel.nyc_taxi.snapshots
    ORDER BY committed_at
""")
snapshots_df.show(truncate=False)

## Part 3: Time Travel: Query Historical Data

### Query Snapshot 1 (June Only)

In [ ]:
snapshot_1 = snapshots_df.collect()[0]['snapshot_id']
print(f"Snapshot 1 ID: {snapshot_1}")

print("\nSnapshot 1 (June data only):")
spark.sql(f"""
    SELECT
        COUNT(*) as trips,
        ROUND(AVG(fare_amount), 2) as avg_fare,
        ROUND(AVG(trip_distance), 2) as avg_distance,
        MIN(tpep_pickup_datetime) as first_pickup,
        MAX(tpep_pickup_datetime) as last_pickup
    FROM polaris.timetravel.nyc_taxi VERSION AS OF {snapshot_1}
""").show(truncate=False)

### Query Snapshot 2 (June + July, Before Cleanup)

In [ ]:
snapshot_2 = snapshots_df.collect()[1]['snapshot_id']
print(f"Snapshot 2 ID: {snapshot_2}")

print("\nSnapshot 2 (June + July, before cleanup):")
spark.sql(f"""
    SELECT
        COUNT(*) as trips,
        ROUND(AVG(fare_amount), 2) as avg_fare,
        ROUND(AVG(trip_distance), 2) as avg_distance
    FROM polaris.timetravel.nyc_taxi VERSION AS OF {snapshot_2}
""").show()

### Compare Current vs Historical

In [ ]:
print("Before cleanup (Snapshot 2) vs After cleanup (current):")
spark.sql(f"""
    SELECT
        'Before cleanup' as version,
        COUNT(*) as trips,
        ROUND(AVG(fare_amount), 2) as avg_fare,
        COUNT(CASE WHEN fare_amount <= 0 THEN 1 END) as bad_fares
    FROM polaris.timetravel.nyc_taxi VERSION AS OF {snapshot_2}
    UNION ALL
    SELECT
        'After cleanup' as version,
        COUNT(*) as trips,
        ROUND(AVG(fare_amount), 2) as avg_fare,
        COUNT(CASE WHEN fare_amount <= 0 THEN 1 END) as bad_fares
    FROM polaris.timetravel.nyc_taxi
""").show()

### Try It: Write Your Own Time Travel Query

Use `VERSION AS OF` to compare a metric of your choice across different snapshots. For example, compare average trip distance, max fare, or total revenue between the June-only snapshot and the current state.

In [ ]:
# Pick two snapshots to compare (snapshot_1 and snapshot_2 are already defined above)
# old_snapshot = ???
# new_snapshot = ???

# Pick a metric to compare: trip_distance, fare_amount, total_amount, passenger_count, etc.
# Write a UNION ALL query using VERSION AS OF to compare the metric across snapshots

# spark.sql(f"""
#     SELECT '???' as version, ??? as my_metric
#     FROM polaris.timetravel.nyc_taxi VERSION AS OF {old_snapshot}
#     UNION ALL
#     SELECT '???' as version, ??? as my_metric
#     FROM polaris.timetravel.nyc_taxi VERSION AS OF {new_snapshot}
# """).show()

## Part 4: Tagging Important Snapshots

### Create a Tag

In [ ]:
current_snapshot = snapshots_df.collect()[-1]['snapshot_id']
print(f"Tagging snapshot {current_snapshot} as 'post-cleanup'")

spark.sql("""
    ALTER TABLE polaris.timetravel.nyc_taxi
    DROP TAG IF EXISTS `post-cleanup`
""")

spark.sql(f"""
    ALTER TABLE polaris.timetravel.nyc_taxi
    CREATE TAG `post-cleanup` AS OF VERSION {current_snapshot}
""")

print("Tag created!")

### Query Using Tag Name

In [ ]:
print("Query using tag 'post-cleanup':")
spark.sql("""
    SELECT COUNT(*) as trips, ROUND(AVG(fare_amount), 2) as avg_fare
    FROM polaris.timetravel.nyc_taxi VERSION AS OF 'post-cleanup'
""").show()

### Tag an Earlier Snapshot

In [ ]:
print(f"Tagging snapshot {snapshot_2} as 'pre-cleanup'")

spark.sql("""
    ALTER TABLE polaris.timetravel.nyc_taxi
    DROP TAG IF EXISTS `pre-cleanup`
""")

spark.sql(f"""
    ALTER TABLE polaris.timetravel.nyc_taxi
    CREATE TAG `pre-cleanup` AS OF VERSION {snapshot_2}
""")

print("Tag created!")

### Check the Refs Metadata Table

The `refs` metadata table lists all named references (branches and tags) and the snapshot each one points to. Every Iceberg table has at least a `main` branch.

In [ ]:
spark.sql("""
    SELECT name, type, snapshot_id
    FROM polaris.timetravel.nyc_taxi.refs
""").show(truncate=False)

## Part 5: Make a Mistake and Rollback

### Make an Intentional Mistake

In [ ]:
good_snapshot = snapshots_df.collect()[-1]['snapshot_id']
print(f"Good snapshot ID: {good_snapshot}")

before_count = spark.sql("SELECT COUNT(*) FROM polaris.timetravel.nyc_taxi").collect()[0][0]

spark.sql("""
    DELETE FROM polaris.timetravel.nyc_taxi
    WHERE tpep_pickup_datetime >= '2023-07-01'
""")

after_count = spark.sql("SELECT COUNT(*) FROM polaris.timetravel.nyc_taxi").collect()[0][0]
print(f"\nMistake: Accidentally deleted all July trips!")
print(f"  Before: {before_count:,} trips")
print(f"  After:  {after_count:,} trips  ({before_count - after_count:,} lost!)")

### Rollback to Previous State

In [ ]:
print(f"Rolling back to snapshot {good_snapshot}...")

spark.sql(f"""
    CALL polaris.system.rollback_to_snapshot(
        table => 'timetravel.nyc_taxi',
        snapshot_id => {good_snapshot}
    )
""")

restored = spark.sql("SELECT COUNT(*) FROM polaris.timetravel.nyc_taxi").collect()[0][0]
print(f"Rollback complete! {restored:,} trips restored")

### Undo the Rollback: Jump Forward Again

Rollback doesn't erase history. Every snapshot still exists, including the "mistake" snapshot. While `rollback_to_snapshot` only moves backward and is the safe choice for undoing mistakes, `set_current_snapshot` can jump to **any** snapshot, forward or backward, and is useful for navigating between arbitrary points in the table's history. The only way to permanently remove snapshots from the metadata is by explicitly expiring them with `expire_snapshots`.

Old snapshots themselves are lightweight (just metadata pointers), but the data files they reference can't be garbage-collected until the snapshots are expired. In practice, you'd run `expire_snapshots` periodically to reclaim storage. We'll cover this in detail in E3.2.

In [ ]:
all_snapshots = spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM polaris.timetravel.nyc_taxi.snapshots
    ORDER BY committed_at
""").collect()

print("All snapshots (nothing was erased):")
for i, s in enumerate(all_snapshots):
    marker = " <-- current (after rollback)" if s['snapshot_id'] == good_snapshot else ""
    if i == 3:
        marker = " <-- the 'mistake'"
    print(f"  {i+1}. {s['snapshot_id']}  {s['operation']:>9}  {s['committed_at']}{marker}")

delete_snapshot = all_snapshots[3]['snapshot_id']
print(f"\nWe can jump forward to the mistake snapshot using set_current_snapshot.")

In [ ]:
print(f"Jumping forward to the 'mistake' snapshot ({delete_snapshot})...")

spark.sql(f"""
    CALL polaris.system.set_current_snapshot(
        table => 'timetravel.nyc_taxi',
        snapshot_id => {delete_snapshot}
    )
""")

mistake_count = spark.sql("SELECT COUNT(*) FROM polaris.timetravel.nyc_taxi").collect()[0][0]
print(f"\nTable is back in the 'mistake' state: {mistake_count:,} trips (July data gone)")

In [ ]:
print(f"Jumping back to the good snapshot ({good_snapshot})...")

spark.sql(f"""
    CALL polaris.system.set_current_snapshot(
        table => 'timetravel.nyc_taxi',
        snapshot_id => {good_snapshot}
    )
""")

restored = spark.sql("SELECT COUNT(*) FROM polaris.timetravel.nyc_taxi").collect()[0][0]
print(f"All trips restored: {restored:,}")

### Try It: Tag and Rollback Practice

Create a tag on a snapshot of your choice, then use that tag to look up the snapshot ID and roll back to it. Can you jump between different tagged snapshots using `set_current_snapshot`?

In [ ]:
# my_tag_name = "???"
# my_tag_snapshot = ???  # paste a snapshot_id from the snapshots table

# List all snapshots to pick one
# spark.sql("""
#     SELECT snapshot_id, committed_at, operation
#     FROM polaris.timetravel.nyc_taxi.snapshots ORDER BY committed_at
# """).show(truncate=False)

# Create a tag on your chosen snapshot
# spark.sql(f"ALTER TABLE polaris.timetravel.nyc_taxi CREATE TAG `{my_tag_name}` AS OF VERSION ???")

# Look up the tag from the refs metadata table and use it to roll back
# tag_id = spark.sql(f"""
#     SELECT snapshot_id FROM polaris.timetravel.nyc_taxi.refs WHERE name = '{my_tag_name}'
# """).collect()[0]['snapshot_id']
# spark.sql(f"CALL polaris.system.rollback_to_snapshot(table => 'timetravel.nyc_taxi', snapshot_id => {tag_id})")

# Verify: how many trips does the table have now?
# print(spark.sql("SELECT COUNT(*) FROM polaris.timetravel.nyc_taxi").collect()[0][0])

# Restore back to the good snapshot when done
# spark.sql(f"CALL polaris.system.set_current_snapshot(table => 'timetravel.nyc_taxi', snapshot_id => {good_snapshot})")

## Part 6: Write-Audit-Publish (WAP)

Write-Audit-Publish lets you stage changes to a table without making them immediately visible to readers. In a traditional database you'd use a transaction for this (`BEGIN`, make changes, review, then `COMMIT` or `ROLLBACK`). But Iceberg writes immutable files to object storage where traditional transactions aren't available. WAP gives you a similar workflow: stage changes, review them, then publish or discard. This is useful for:
- Validating data quality before publishing
- Reviewing changes in a staging environment
- Rolling back staged changes that fail validation

The workflow is:
1. **Enable WAP** on the table
2. **Write** with a `wap.id`. Changes are stored but invisible to normal queries
3. **Audit** the staged snapshot using `VERSION AS OF`
4. **Publish** to make changes visible, or **discard** to remove them

In [ ]:
spark.sql("""
    ALTER TABLE polaris.timetravel.nyc_taxi
    SET TBLPROPERTIES ('write.wap.enabled' = 'true')
""")

print("WAP enabled on nyc_taxi table")

### Stage a WAP Write

Setting `spark.wap.id` in the Spark configuration tells Iceberg to treat any subsequent write as a staged WAP commit. This is a Spark-specific integration. Iceberg's Spark plugin reads `spark.wap.id` and uses it to tag the resulting snapshot. Other engines may implement WAP differently or not at all. The write still creates a snapshot with data files, but that snapshot is not set as the table's current snapshot so it remains invisible to normal queries. After the write, unset the property so future operations will not use this wap.id.

In [ ]:
pre_wap_count = spark.sql("SELECT COUNT(*) FROM polaris.timetravel.nyc_taxi").collect()[0][0]

spark.conf.set("spark.wap.id", "zero-distance-cleanup")

spark.sql("""
    DELETE FROM polaris.timetravel.nyc_taxi
    WHERE trip_distance <= 0
""")

spark.conf.unset("spark.wap.id")

print("Staged cleanup of zero-distance trips with wap.id = 'zero-distance-cleanup'")

In [ ]:
prod_count = spark.sql("SELECT COUNT(*) FROM polaris.timetravel.nyc_taxi").collect()[0][0]
print(f"Before WAP write: {pre_wap_count:,} trips")
print(f"After WAP write:  {prod_count:,} trips")
print(f"\nProduction is unchanged; the staged delete is not visible.")

### Audit the Staged Snapshot

In [ ]:
wap_snapshot_id = spark.sql("""
    SELECT snapshot_id
    FROM polaris.timetravel.nyc_taxi.snapshots
    WHERE summary['wap.id'] = 'zero-distance-cleanup'
""").collect()[0]['snapshot_id']

wap_count = spark.sql(f"""
    SELECT COUNT(*) FROM polaris.timetravel.nyc_taxi VERSION AS OF {wap_snapshot_id}
""").collect()[0][0]

print(f"WAP Snapshot ID: {wap_snapshot_id}")
print(f"\nProduction:  {prod_count:,} trips")
print(f"WAP staged:  {wap_count:,} trips")
print(f"Would remove: {prod_count - wap_count:,} zero-distance trips")

### Publishing WAP Changes

After auditing, you have two options:
- **Publish**: `CALL system.publish_changes(...)` makes the staged snapshot the current one
- **Discard**: `CALL system.expire_snapshots(...)` removes the staged snapshot entirely

In [ ]:
spark.sql("""
    CALL polaris.system.publish_changes(
        table => 'polaris.timetravel.nyc_taxi',
        wap_id => 'zero-distance-cleanup'
    )
""")

published_count = spark.sql("SELECT COUNT(*) FROM polaris.timetravel.nyc_taxi").collect()[0][0]
print(f"WAP changes published! {published_count:,} trips in production")

### Discarding an Unwanted WAP Commit

If a staged change fails validation, you can discard it instead of publishing.

In [ ]:
spark.conf.set("spark.wap.id", "bad-cleanup")

spark.sql("""
    DELETE FROM polaris.timetravel.nyc_taxi
    WHERE total_amount < 50
""")

spark.conf.unset("spark.wap.id")

print("Staged a bad cleanup with wap.id = 'bad-cleanup'")

In [ ]:
bad_snapshot_id = spark.sql("""
    SELECT snapshot_id
    FROM polaris.timetravel.nyc_taxi.snapshots
    WHERE summary['wap.id'] = 'bad-cleanup'
""").collect()[0]['snapshot_id']

bad_count = spark.sql(f"""
    SELECT COUNT(*) FROM polaris.timetravel.nyc_taxi VERSION AS OF {bad_snapshot_id}
""").collect()[0][0]

current_count = spark.sql("SELECT COUNT(*) FROM polaris.timetravel.nyc_taxi").collect()[0][0]

print(f"Bad WAP Snapshot ID: {bad_snapshot_id}")
print(f"\nProduction:  {current_count:,} trips")
print(f"WAP staged:  {bad_count:,} trips")
print(f"Would remove: {current_count - bad_count:,} trips, way too many!")

print("\nThe bad snapshot is visible in the snapshots table:")
spark.sql("""
    SELECT snapshot_id, operation, summary['wap.id'] as wap_id
    FROM polaris.timetravel.nyc_taxi.snapshots
    ORDER BY committed_at
""").show(truncate=False)

In [ ]:
spark.sql(f"""
    CALL polaris.system.expire_snapshots(
        table => 'polaris.timetravel.nyc_taxi',
        snapshot_ids => ARRAY({bad_snapshot_id})
    )
""")

safe_count = spark.sql("SELECT COUNT(*) FROM polaris.timetravel.nyc_taxi").collect()[0][0]
print(f"Bad WAP snapshot discarded! Production unchanged: {safe_count:,} trips")

print("\nThe bad snapshot has been removed from the snapshots table:")
spark.sql("""
    SELECT snapshot_id, operation, summary['wap.id'] as wap_id
    FROM polaris.timetravel.nyc_taxi.snapshots
    ORDER BY committed_at
""").show(truncate=False)

In [ ]:
spark.sql("""
    ALTER TABLE polaris.timetravel.nyc_taxi
    SET TBLPROPERTIES ('write.wap.enabled' = 'false')
""")

print("WAP disabled on nyc_taxi table")

### Try It: Stage Your Own WAP Write

Re-enable WAP and stage your own data quality cleanup. Try removing trips with unrealistic values (e.g., `trip_distance > 200` or `passenger_count = 0`). Audit the staged snapshot, then decide whether to publish or discard.

In [ ]:
# my_wap_id_name = "???"
# my_filter = "???"  # e.g. trip_distance > 200, passenger_count = 0, etc.

# Re-enable WAP
# spark.sql("ALTER TABLE polaris.timetravel.nyc_taxi SET TBLPROPERTIES ('write.wap.enabled' = 'true')")

# Stage a WAP write with your cleanup rule
# spark.conf.set("spark.wap.id", my_wap_id_name)
# spark.sql(f"DELETE FROM polaris.timetravel.nyc_taxi WHERE {my_filter}")
# spark.conf.unset("spark.wap.id")

# Audit: find the staged snapshot and check how many rows would be removed
# my_wap_snap = spark.sql(f"""
#     SELECT snapshot_id FROM polaris.timetravel.nyc_taxi.snapshots
#     WHERE summary['wap.id'] = ???
# """).collect()[0]['snapshot_id']
# wap_rows = spark.sql(f"SELECT COUNT(*) FROM polaris.timetravel.nyc_taxi VERSION AS OF ???").collect()[0][0]
# prod_rows = spark.sql("SELECT COUNT(*) FROM polaris.timetravel.nyc_taxi").collect()[0][0]
# print(f"Production: {prod_rows:,}  |  After cleanup: {wap_rows:,}  |  Would remove: {prod_rows - wap_rows:,}")

# Publish or discard? Pick one:
# spark.sql(f"CALL polaris.system.publish_changes(table => 'polaris.timetravel.nyc_taxi', wap_id => ???)")
# spark.sql(f"CALL polaris.system.expire_snapshots(table => 'polaris.timetravel.nyc_taxi', snapshot_ids => ARRAY(???))")

# Check the final state of the table
# final_count = spark.sql("SELECT COUNT(*) FROM polaris.timetravel.nyc_taxi").collect()[0][0]
# print(f"Final table count: {final_count:,}")
# spark.sql("""
#     SELECT snapshot_id, operation, summary['wap.id'] as wap_id
#     FROM polaris.timetravel.nyc_taxi.snapshots ORDER BY committed_at
# """).show(truncate=False)

# Disable WAP when done
# spark.sql("ALTER TABLE polaris.timetravel.nyc_taxi SET TBLPROPERTIES ('write.wap.enabled' = 'false')")

## Part 7: Branching and Merging

Iceberg branches work like Git branches: they are lightweight metadata pointers, not data copies. Creating a branch is instant because it just points to the same snapshot as main. New writes to the branch create new files, but all existing files are shared with main. No data is duplicated.

Use cases:
- **Experimentation**: Test schema changes or data transformations in isolation
- **ETL staging**: Write new data to a branch, validate, then merge to main
- **A/B testing**: Maintain different versions of data for comparison

### Create a Branch

In [ ]:
spark.sql("""
    ALTER TABLE polaris.timetravel.nyc_taxi
    DROP BRANCH IF EXISTS `august_load`
""")

spark.sql("""
    ALTER TABLE polaris.timetravel.nyc_taxi
    CREATE BRANCH august_load
""")

print("Branch 'august_load' created!")

### Write to the Branch

In Spark, you address a branch by appending `.branch_<name>` to the table name (e.g., `nyc_taxi.branch_august_load`). This is a Spark-specific convention. Other engines may use different syntax like `AT BRANCH`.

In [ ]:
spark.sql("""
    INSERT INTO polaris.timetravel.nyc_taxi.branch_august_load
    SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-08.parquet`
""")

print("Added August 2023 data to the 'august_load' branch")

In [ ]:
branch_count = spark.sql("""
    SELECT COUNT(*) FROM polaris.timetravel.nyc_taxi.branch_august_load
""").collect()[0][0]

main_count = spark.sql("""
    SELECT COUNT(*) FROM polaris.timetravel.nyc_taxi
""").collect()[0][0]

print(f"Branch 'august_load': {branch_count:,} trips (June + July + August)")
print(f"Main branch:          {main_count:,} trips (June + July only, unchanged)")

### Merge Branch to Main

Use `fast_forward` to merge the branch back into main. The branch must be a direct descendant of main's current snapshot. If main has diverged (someone else wrote to it after you branched), the fast-forward will fail and you'll need to recreate the branch from the updated main. In practice, keep branches short-lived to avoid divergence.

In [ ]:
spark.sql("""
    CALL polaris.system.fast_forward(
        table => 'timetravel.nyc_taxi',
        branch => 'main',
        to => 'august_load'
    )
""").show()

print("Branch 'august_load' merged into main!")

In [ ]:
merged_count = spark.sql("SELECT COUNT(*) FROM polaris.timetravel.nyc_taxi").collect()[0][0]
print(f"Main branch after merge: {merged_count:,} trips (June + July + August)")

print("\nAll refs:")
spark.sql("""
    SELECT name, type, snapshot_id
    FROM polaris.timetravel.nyc_taxi.refs
""").show(truncate=False)

### Try It: Create Your Own Branch

Create a new branch, make a data change on it (e.g., filter out rows, insert new records, or update values), then compare the branch to main. Decide whether to merge or drop it.

In [ ]:
# my_branch = "???"

# Create the branch
# spark.sql(f"ALTER TABLE polaris.timetravel.nyc_taxi CREATE BRANCH {my_branch}")

# Make a data change on the branch. Try a DELETE, INSERT, or UPDATE
# Hint: address the branch as polaris.timetravel.nyc_taxi.branch_<name>
# spark.sql(f"""
#     ???
# """)

# Compare branch vs main
# branch_count = spark.sql(f"SELECT COUNT(*) FROM polaris.timetravel.nyc_taxi.branch_{my_branch}").collect()[0][0]
# main_count = spark.sql("SELECT COUNT(*) FROM polaris.timetravel.nyc_taxi").collect()[0][0]
# print(f"Main: {main_count:,}  |  Branch: {branch_count:,}")

# Merge or drop? Pick one:
# spark.sql(f"CALL polaris.system.fast_forward(table => 'timetravel.nyc_taxi', branch => 'main', to => ???)")
# spark.sql(f"ALTER TABLE polaris.timetravel.nyc_taxi DROP BRANCH `{my_branch}`")

## Part 8: Audit Trail with Snapshots

**A note on added/deleted counts**: You may notice that the `overwrite` operations below show millions of rows "added" and "deleted" even though only ~65K bad-fare rows were removed. This might seem wildly expensive compared to a traditional database's row-level deletes, and it can be. This is because Iceberg's default write mode, copy-on-write (COW), rewrites entire data files when any row in them changes. Deleting 65K rows spread across many files means rewriting all those files, so the added/deleted counts reflect the full file contents, not just the changed rows. Exercise 3.1 covers an alternative called merge-on-read (MOR) that avoids rewriting entire files by logging changes separately.

In [ ]:
print("Complete snapshot history:")
spark.sql("""
    SELECT
        snapshot_id,
        committed_at,
        operation,
        summary['added-records'] as added,
        summary['deleted-records'] as deleted
    FROM polaris.timetravel.nyc_taxi.snapshots
    ORDER BY committed_at
""").show(truncate=False)

## Cleanup

In [ ]:
# Optional: Drop table to start fresh
# spark.sql("DROP TABLE IF EXISTS polaris.timetravel.nyc_taxi")
# print("Table dropped!")